# Dataset formats

> A registry of on-disk dataset formats, each with a streaming writer and a reader.

In [ ]:
#| default_exp data.format

In [ ]:
#| hide
from nbdev.showdoc import *

Same design as stable-worldmodel's format registry: a `Format` ties a name to a `detect`
check, a writer and a reader. New formats (HDF5, Lance, ...) only need a `Format` subclass
decorated with `@register_format`.

An **episode** is a dict of numpy arrays, with time first (see `World.collect`).

In [ ]:
#| export
from __future__ import annotations

from pathlib import Path
from typing import Any, Iterable, Protocol, runtime_checkable

import numpy as np

In [ ]:
#| export
WRITE_MODES = ('append', 'overwrite', 'error')
FORMATS: dict[str, type[Format]] = {}


@runtime_checkable
class Writer(Protocol):
    "Streaming episode writer, used as a context manager."
    def __enter__(self) -> Writer: ...
    def __exit__(self, *exc) -> None: ...
    def write_episode(self, episode: dict[str, np.ndarray]) -> None: ...
    def write_episodes(self, episodes: Iterable[dict[str, np.ndarray]]) -> None: ...


@runtime_checkable
class EpisodeReader(Protocol):
    "Random access to the episodes of a dataset."
    metadata: dict[str, Any]
    @property
    def num_episodes(self) -> int: ...
    def episode(self, index: int, keys: Iterable[str] | None = None) -> dict[str, np.ndarray]: ...


class Format:
    "An on-disk dataset format: subclasses set `name` and implement the methods they support."
    name: str = ''

    @classmethod
    def detect(cls, path: str | Path) -> bool:
        "Whether `path` holds a dataset in this format."
        raise NotImplementedError

    @classmethod
    def open_writer(cls, path: str | Path, mode: str = 'append',
                    metadata: dict[str, Any] | None = None) -> Writer:
        raise NotImplementedError(f"Format {cls.name!r} cannot be written")

    @classmethod
    def open_reader(cls, path: str | Path) -> EpisodeReader:
        raise NotImplementedError(f"Format {cls.name!r} cannot be read")

In [ ]:
#| export
def register_format(cls: type[Format]) -> type[Format]:
    "Class decorator registering a :class:`Format` under its `name`."
    if not cls.name:
        raise ValueError(f"{cls.__name__} must set a `name`")
    FORMATS[cls.name] = cls
    return cls


def list_formats() -> list[str]:
    return sorted(FORMATS)


def get_format(name: str) -> type[Format]:
    if name not in FORMATS:
        raise ValueError(f"Unknown dataset format {name!r}, available: {list_formats()}")
    return FORMATS[name]


def validate_write_mode(mode: str) -> str:
    if mode not in WRITE_MODES:
        raise ValueError(f"Write mode must be one of {WRITE_MODES}, got {mode!r}")
    return mode


def load_episodes(path: str | Path, format: str | None = None) -> EpisodeReader:
    "Open the dataset at `path` for reading, detecting its format unless `format` is given."
    if format is None:
        matches = [fmt for fmt in FORMATS.values() if fmt.detect(path)]
        if not matches:
            raise ValueError(f"No dataset found at {path} (formats: {list_formats()})")
        return matches[0].open_reader(path)
    return get_format(format).open_reader(path)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()